# Intraday Model

In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from boosting_utils import (
    FEATURES, baseline_models, ensure_output_dir, fit_best_classifier, fit_regressor,
    long_short_backtest, make_boosting_panel, make_intraday_features, model_interpretation,
    score_quantiles, tune_gbm
)

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
OUTPUT = ensure_output_dir()
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score


In [2]:
intraday = make_intraday_features(seed=131)
features = ['momentum_6', 'momentum_18', 'volatility_18']
cutoff = int(len(intraday) * 0.75)
train, test = intraday.iloc[:cutoff], intraday.iloc[cutoff:]
model = GradientBoostingClassifier(n_estimators=80, learning_rate=0.04, max_depth=2, random_state=42)
model.fit(train[features], train['target_positive'])
test = test.copy()
test['score'] = model.predict_proba(test[features])[:, 1]
roc = roc_auc_score(test['target_positive'], test['score'])
test.to_parquet(OUTPUT / 'intraday_model_scores.parquet', index=False)
pd.Series({'roc_auc': roc, 'rows': len(test)}).to_frame('value')

,value
roc_auc,0.4666
rows,189.0000
